# Studying the final neutron star population

In [ ]:
import astropy.coordinates as coord
import astropy.units as u
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl
from scipy.integrate import quad
from pypopsyn.simulator.configuration import cfg

# Set `usetex=False' if you do not have LaTeX installed.
rc('text', usetex=False)
rc('font', family='serif')
mpl.rcParams['text.latex.preamble'] = [r"\usepackage{amsmath}"]

In [ ]:
rcParams["mathtext.fontset"] = "stix"
# rcParams["font.family"] = "Liberation serif"
rcParams["font.size"] = "22"
# rcParams['font.weight']='bold'
rcParams["figure.figsize"] = "8.0, 8.0"
rcParams["figure.autolayout"] = "False"

rcParams["axes.linewidth"] = "1.7"
rcParams["axes.labelpad"] = "15.0"
rcParams["axes.titlepad"] = "15.0"

rcParams["xtick.direction"] = "in"
rcParams["xtick.top"] = True
rcParams["xtick.major.pad"] = "10.0"
rcParams["xtick.minor.pad"] = "10.0"
rcParams["xtick.major.size"] = "10.0"
rcParams["xtick.major.width"] = "1.7"
rcParams["xtick.minor.size"] = "5.0"
rcParams["xtick.minor.width"] = "1.7"
rcParams["xtick.labelsize"] = "25"

rcParams["ytick.direction"] = "in"
rcParams["ytick.right"] = True
rcParams["ytick.major.pad"] = "10.0"
rcParams["ytick.minor.pad"] = "10.0"
rcParams["ytick.major.size"] = "10.0"
rcParams["ytick.major.width"] = "1.7"
rcParams["ytick.minor.size"] = "5.0"
rcParams["ytick.minor.width"] = "1.7"
rcParams["ytick.labelsize"] = "25"

Select an `final_population.pkl.gz` file to import:

In [ ]:
data = pd.read_pickle("../data/final_population.pkl.gz", compression="gzip")
data.head()

In [ ]:
x = data["x"]["[kpc]"].to_numpy()
y = data["y"]["[kpc]"].to_numpy()
z = data["z"]["[kpc]"].to_numpy()
RA = data["RA"]["[deg]"].to_numpy()
DEC = data["DEC"]["[deg]"].to_numpy()
v_RA = data["v_RA"]["[mas/yr]"].to_numpy()
v_DEC = data["v_DEC"]["[mas/yr]"].to_numpy()
B = data["B"]["[G]"].to_numpy()
chi = data["chi"]["[rad]"].to_numpy()
P = data["P"]["[s]"].to_numpy()
P_dot = data["P_dot"]["[s/s]"].to_numpy()

## Positional information

Top view of the galactic plane

In [ ]:
fig, ax = plt.subplots()

ax.plot(
    x,
    y,
    linestyle="None",
    marker="o",
    color="blue",
    markersize=1,
    alpha=0.2,
    rasterized=False
)

ax.plot(0.0, 8.5, marker="o", color="gold", markersize=6)
ax.set_xlabel(r"$x$ [kpc]")
ax.set_ylabel(r"$y$ [kpc]")

plt.show()

Zoomed-in top view of the galactic plane

In [ ]:
fig, ax = plt.subplots()

ax.plot(
    x,
    y,
    linestyle="None",
    marker="o",
    color="blue",
    markersize=1,
    alpha=0.3,
    rasterized=False
)

ax.plot(0.0, 8.5, marker="o", color="gold", markersize=6)
ax.set_xlabel(r"$x$ [kpc]")
ax.set_ylabel(r"$y$ [kpc]")
ax.set_xlim(-20.0, 20.0)
ax.set_ylim(-20.0, 20.0)

plt.show()

Side view of the galactic plane

In [ ]:
fig, ax = plt.subplots()

ax.plot(
    x,
    z,
    linestyle="None",
    marker="o",
    color="blue",
    markersize=1,
    alpha=0.3,
    rasterized=True,
)

ax.plot(0.0, 0.02, marker="o", color="gold", markersize=6)
ax.set_xlabel(r"$x$ [kpc]")
ax.set_ylabel(r"$z$ [kpc]")

plt.show()

Zoomed-in side view of the galactic plane

In [ ]:
fig, ax = plt.subplots()

ax.plot(
    x,
    z,
    linestyle="None",
    marker="o",
    color="blue",
    markersize=1,
    alpha=0.3,
    rasterized=True,
)

ax.plot(0.0, 0.02, marker="o", color="gold", markersize=6)
ax.set_xlabel(r"$x$ [kpc]")
ax.set_ylabel(r"$z$ [kpc]")
ax.set_xlim(-20.0, 20.0)
ax.set_ylim(-20.0, 20.0)

plt.show()

Histrogramming the pulsars position and comparing to underlying initial position PDF

In [ ]:
def pdf_r(r):
    "pdf from the Milky Way stellar surface density (Yusifov & Küçük 2004)"
    A = 37.6  # [stars kpc^-2]
    R1 = 0.55  # +-0.1 [kpc]
    Rsun = 8.5
    a = 1.64  # +-0.11
    b = 4.01  # +-0.24
    rho = (
        A
        * ((r + R1) / (Rsun + R1)) ** a
        * np.exp(-b * ((r - Rsun) / (Rsun + R1)))
    )
    pdf_r = 2 * np.pi * r * rho
    return pdf_r

For normalization purposes, determine the area underneath the theoretical PDF curve:

In [ ]:
pdf_area = quad(pdf_r, 0, 100)[0]
print(pdf_area)

In [ ]:
r = np.sqrt(x**2 + y**2)
r_bins = np.linspace(0.0, 30.0, 51)

In [ ]:
fig, ax = plt.subplots()

ax.hist(
    r,
    bins=r_bins,
    histtype="step",
    edgecolor="blue",
    lw=4,
    alpha=0.5,
    label="simulation evolved",
    density=True
)
ax.plot(
    r_bins,
    pdf_r(r_bins) / pdf_area,
    linestyle="-",
    lw=4,
    color="red",
    alpha=0.5,
    label="initial YK04",
)
plt.xlabel(r"$r$ [kpc]")
plt.ylabel(r"normalized radial PDF")
plt.xlim(0.0, 30.0)
plt.legend(frameon=False, loc=0)

plt.show()

## Comparison with the ATNF catalogue

Plotting the distribution of simulated pulsars in RA and DEC in the ICRS (International Celestial Reference System) frame and comparing with neutron stars in the ATNF catalogue.

In [ ]:
data_atnf = pd.read_csv("../data/ATNF_pulsars_20-05-2020.csv", delimiter=',', header=[0,1])
data_atnf.head()

In [ ]:
RA_atnf = coord.Angle(data_atnf["RAJ"].values, unit=u.hourangle)
DEC_atnf = coord.Angle(data_atnf["DECJ"].values, unit=u.deg)

# Convert RA and DEC in degrees and transform to np.array
icrs_coord = coord.ICRS(ra=RA_atnf, dec=DEC_atnf)

RA_atnf = np.array(icrs_coord.ra.degree / u.deg)
DEC_atnf = np.array(icrs_coord.dec.degree / u.deg)

In [ ]:
RA_galcen = 266.4
DEC_galcen = -29.0

fig, ax = plt.subplots()

ax.plot(
    RA, 
    DEC, 
    linestyle="None",
    marker="o",
    color="blue",
    markersize=1,
    alpha=0.3,
    rasterized=True,
)
ax.plot(
    RA_atnf, 
    DEC_atnf, 
    linestyle="None",
    marker="o",
    color="red",
    markersize=3,
    alpha=0.3,
    rasterized=True,
)

ax.plot(RA_galcen, DEC_galcen, marker="X", color="gold", markersize=10)
ax.set_xlim(0., 360.)
ax.set_ylim(-90., 90.)
ax.set_xlabel('RA [deg]')
ax.set_ylabel('DEC [deg]')

plt.show()

Histrogramming the simulated pulsars RA and DEC coordinate position and comparing with the observed neutron stars in the ATNF catalogue.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 12))

RA_bins = np.linspace(0., 360., 50)
DEC_bins = np.linspace(-90., 90., 50)

ax1.hist(
    RA,
    bins=RA_bins,
    histtype="step",
    edgecolor="blue",
    lw=4,
    alpha=1,
    label=r"simulated",
    density=1,
)
ax1.hist(
    RA_atnf,
    bins=RA_bins,
    histtype="stepfilled",
    edgecolor="red",
    lw=0.01,
    alpha=0.5,
    label=r"observed",
    density=1,
)
ax1.set_xlabel(r"RA [deg]")
ax1.set_ylabel(r"normalized count")
ax1.legend(frameon=False, loc=0)

ax2.hist(
    DEC,
    bins=DEC_bins,
    histtype="step",
    edgecolor="blue",
    lw=4,
    alpha=1,
    label=r"simulated",
    density=1,
)
ax2.hist(
    DEC_atnf,
    bins=DEC_bins,
    histtype="stepfilled",
    edgecolor="red",
    lw=0.01,
    alpha=0.5,
    label=r"observed",
    density=1,
)
ax2.set_xlabel(r"DEC [deg]")
ax2.set_ylabel(r"normalized count")
ax2.legend(frameon=False, loc=0)

plt.show()

## Proper velocity comparison

Histrogramming the simulated pulsars angular proper velocity in RA and DEC and comparing with the proper velocity of the observed neutron stars.

In [ ]:
data_pm = pd.read_csv("../data/PSRs_prop_motion_22-05-2020.csv", header=[0,1])
data_pm.head()

In [ ]:
v_RA_pm = data_pm["PMRA"]["[mas / yr]"].to_numpy()
v_DEC_pm = data_pm["PMDEC"]["[mas / yr]"].to_numpy()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

x_bins = np.linspace(-50, 50, 50)

ax1.hist(
    v_RA,
    bins=x_bins,
    histtype="step",
    edgecolor="blue",
    lw=4,
    alpha=1,
    label=r"simulated",
    density=1,
)
ax1.hist(
    v_RA_pm,
    bins=x_bins,
    histtype="stepfilled",
    edgecolor="red",
    lw=0.01,
    alpha=0.5,
    label=r"observed",
    density=1,
)
ax1.set_xlabel(r"$v_{\rm RA}$ [mas yr$^{-1}$]")
ax1.set_ylabel(r"normalized count")
ax1.legend(frameon=False, loc=0)

ax2.hist(
    v_DEC,
    bins=x_bins,
    histtype="step",
    edgecolor="blue",
    lw=4,
    alpha=1,
    label=r"simulated",
    density=1,
)
ax2.hist(
    v_DEC_pm,
    bins=x_bins,
    histtype="stepfilled",
    edgecolor="red",
    lw=0.01,
    alpha=0.5,
    label=r"observed",
    density=1,
)
ax2.set_xlabel(r"$v_{\rm DEC}$ [mas yr$^{-1}$]")
ax2.set_ylabel(r"normalized count")
ax2.legend(frameon=False, loc=0)

plt.show()

## Magneto-rotational information

Histogramming the periods, magnetic fields and misalignment angles

In [ ]:
def Gaussian(x, mean, sigma):
    y = (
        1
        / (sigma * np.sqrt(2 * np.pi))
        * np.exp(-((x - mean) ** 2) / (2 * sigma ** 2))
    )
    return y

In [ ]:
P_bins = np.linspace(-0.5, 160, 2001)
print(max(P), min(P))

In [ ]:
pdf_P_initial_area = quad(Gaussian, 0, 100, args=(cfg["P_initial_mean"], cfg["P_initial_sigma"]))[0]
print(pdf_P_initial_area)

In [ ]:
fig, ax = plt.subplots()

ax.hist(
    P,
    bins=P_bins,
    histtype="step",
    edgecolor="blue",
    lw=4,
    alpha=0.5,
    label="simulation final",
    density=True
)
ax.plot(P_bins, 
        Gaussian(
            P_bins, 
            cfg["P_initial_mean"], 
            cfg["P_initial_sigma"]
        ),
        linestyle="-",
        lw=4,
        color="red",
        alpha=0.5,
        label="theoretical initial",
)
ax.plot(P_bins[P_bins > 0], 
        Gaussian(
            P_bins[P_bins > 0], 
            cfg["P_initial_mean"], 
            cfg["P_initial_sigma"]
        ) / pdf_P_initial_area,
        linestyle="-",
        lw=4,
        color="green",
        alpha=0.5,
        label="rescaled initial",
)
plt.xlabel(r"$P$ [s]")
plt.ylabel(r"normalized PDF")
plt.xlim(-0.5, 10.0)
plt.legend(frameon=False, loc=1)

plt.show()

In [ ]:
B_log10_bins = np.linspace(9.0, 17.0, 101)
print(max(np.log10(B)), min(np.log10(B)))

In [ ]:
fig, ax = plt.subplots()

ax.hist(
    np.log10(B),
    bins=B_log10_bins,
    histtype="step",
    edgecolor="blue",
    lw=4,
    alpha=0.5,
    label="simulation final",
    density=True
)
ax.plot(B_log10_bins, 
        Gaussian(
            B_log10_bins, 
            cfg["B_initial_log10_mean"], 
            cfg["B_initial_log10_sigma"]
        ),
        linestyle="-",
        lw=4,
        color="red",
        alpha=0.5,
        label="theoretical initial",
)
plt.xlabel(r"log$_{10} B$ [G]")
plt.ylabel(r"normalized PDF")
plt.xlim(9., 17.0)
plt.legend(frameon=False, loc=1)

plt.show()

In [ ]:
chi_bins = np.linspace(0, np.pi / 2, 301)
print(max(chi), min(chi))

In [ ]:
fig, ax = plt.subplots()

ax.hist(
    chi,
    bins=chi_bins,
    histtype="step",
    edgecolor="blue",
    lw=4,
    alpha=0.5,
    label="simulation final",
    density=True
)
ax.plot(chi_bins, 
        np.sin(chi_bins),
        linestyle="-",
        lw=4,
        color="red",
        alpha=0.5,
        label="theoretical initial",
)
plt.xlabel(r"$\chi$ [rad]")
plt.ylabel(r"normalized PDF")
plt.xlim(0., np.pi / 2)
plt.legend(frameon=False, loc=2)

plt.show()

Plotting the PPdot diagram of the final pulsar population, representin a snap-shot at the current time.

In [ ]:
len(P[::40])

In [ ]:
fig, ax = plt.subplots()

ax.loglog(P[::40], P_dot[::40], '.', color='gray', ms=6)

plt.show()

## Time-evolution in the PPdot diagram

In [ ]:
import pypopsyn.simulator.magneto_rotational_physics.magneto_rotational_evolution as mre
import pypopsyn.simulator.magneto_rotational_physics.period_derivative as pdv
from scipy.integrate import solve_ivp

Consider two pulsars which differ only by their initial fields.

In [ ]:
cfg["NS_number"] = 2
cfg["t_age_max"] = 3e7
cfg["time_step"] = 1e1

B_initial_test = np.array([1e12, 1e14])
chi_initial_test = np.array([np.pi / 3, np.pi / 3])
P_initial_test = np.array([0.01, 0.01])
t_age_test = np.array([cfg["t_age_max"], cfg["t_age_max"]])

Determine their initial period derivatives.

In [ ]:
P_dot_initial_test = np.zeros(2)

P_dot_initial_test[0] = pdv.period_derivative(
    B_initial_test[0],
    chi_initial_test[0],
    P_initial_test[0]
)

P_dot_initial_test[1] = pdv.period_derivative(
    B_initial_test[1],
    chi_initial_test[1],
    P_initial_test[1]
)

Obtaint the evolution in time for a single object:

In [ ]:
def magneto_rotational_evolution_tracked_single_object(
    B_initial,
    chi_initial,
    P_initial,
    t_age,
):
    
    # Initialization of the time grid.
    time_grid = np.append(np.arange(0, t_age, cfg["time_step"]), t_age)

    # Initial conditions for the three parameters.
    y_initial = np.array([B_initial, chi_initial, P_initial])

    evol_output = solve_ivp(
        mre.combined_derivatives,
        t_span=[0, t_age],
        y0=y_initial,
        method="RK45",
        t_eval=time_grid,
        args=(B_initial,),
    ).y

    return evol_output

In [ ]:
B_A, chi_A, P_A = magneto_rotational_evolution_tracked_single_object(
    B_initial_test[0],
    chi_initial_test[0],
    P_initial_test[0],
    t_age_test[0],
)

B_B, chi_B, P_B = magneto_rotational_evolution_tracked_single_object(
    B_initial_test[1],
    chi_initial_test[1],
    P_initial_test[1],
    t_age_test[1],
)

In [ ]:
period_derivative_vect = np.vectorize(pdv.period_derivative)

P_dot_A = period_derivative_vect(B_A, chi_A, P_A)
P_dot_B = period_derivative_vect(B_B, chi_B, P_B)

As a comparison, determine the period derivatives at the current time with the package functions.

In [ ]:
B_final_test, chi_final_test, P_final_test = mre.magneto_rotational_evolution(
    B_initial_test,
    chi_initial_test,
    P_initial_test, 
    t_age_test
)

In [ ]:
P_dot_final_test = np.zeros(2)

P_dot_final_test[0] = pdv.period_derivative(
    B_final_test[0],
    chi_final_test[0],
    P_final_test[0]
)

P_dot_final_test[1] = pdv.period_derivative(
    B_final_test[1],
    chi_final_test[1],
    P_final_test[1]
)

In [ ]:
fig, ax = plt.subplots()

ax.loglog(P[::10], P_dot[::10], '.', color='gray', ms=6)
ax.loglog(P_A, P_dot_A, '-', color='orange')
ax.loglog(P_B, P_dot_B, '-', color='red')
ax.loglog(P_initial_test, P_dot_initial_test, 'o', color='blue', ms=8)
ax.loglog(P_final_test, P_dot_final_test, 'o', color='green', ms=8)

ax.set_xlim(5e-3, 1e2)

plt.show()

This again shows how sensitive the code is to the oldest neutron star in the sample.

THE CUT-OFF AT THE BOTTOM LOOKS NOT RIGHT???? ALSO SOME OF THE PERIODS SEEM RATHER LARGE.

Plot the three quantities as a function of time.

In [ ]:
time_grid = np.append(np.arange(0, cfg["t_age_max"], cfg["time_step"]), cfg["t_age_max"])

In [ ]:
fig, ax = plt.subplots()

ax.loglog(time_grid, B_A, '-', color='orange', lw=3, label=r"$10^{12}$ [G]")
ax.loglog(time_grid, B_B, '-', color='red', lw=3, label=r"$10^{14}$ [G]")

ax.set_xlim(1, 5e7)

plt.xlabel(r"$t$ [yr]")
plt.ylabel(r"$B$ [G]")
plt.legend(loc=3)

plt.show()

In [ ]:
fig, ax = plt.subplots()

ax.loglog(time_grid, chi_A, '-', color='orange', lw=3, label=r"$10^{12}$ [G]")
ax.loglog(time_grid, chi_B, '-', color='red', lw=3, label=r"$10^{14}$ [G]")

ax.set_xlim(1, 5e7)

plt.xlabel(r"$t$ [yr]")
plt.ylabel(r"$\chi$ [rad]")
plt.legend(loc=3)

plt.show()

In [ ]:
fig, ax = plt.subplots()

ax.loglog(time_grid, P_A, '-', color='orange', lw=3, label=r"$10^{12}$ [G]")
ax.loglog(time_grid, P_B, '-', color='red', lw=3, label=r"$10^{14}$ [G]")

ax.set_xlim(1, 5e7)

plt.xlabel(r"$t$ [yr]")
plt.ylabel(r"$P$ [s]")
plt.legend(loc=2)

plt.show()